# 02 Baseline Evaluation

Self-contained dictionary baseline with BLEU evaluation.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import sacrebleu

try:
    import sentencepiece as spm
except ImportError:
    spm = None

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TOKENIZED_DIR = PROJECT_ROOT / 'Data' / 'tokenized'
VOCAB_DIR = PROJECT_ROOT / 'Data' / 'vocab'
OUTPUT_TABLES = PROJECT_ROOT / 'outputs' / 'tables'
OUTPUT_TRANSLATIONS = PROJECT_ROOT / 'outputs' / 'translations'
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT_TRANSLATIONS.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'train_src': TOKENIZED_DIR / 'train.ar.bpe',
    'train_tgt': TOKENIZED_DIR / 'train.en.bpe',
    'test_src': TOKENIZED_DIR / 'test.ar.bpe',
    'test_tgt': TOKENIZED_DIR / 'test.en.bpe',
    'src_spm': VOCAB_DIR / 'sp_ar.model',
    'tgt_spm': VOCAB_DIR / 'sp_en.model',
    'eval_limit': 1000,
}
CONFIG

In [2]:
def read_lines(path):
    return Path(path).read_text(encoding='utf-8').splitlines()

def load_vocab(path):
    tokens = []
    with open(path, 'r', encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                tokens.append(line.split('\t', 1)[0])
    return tokens, {token: idx for idx, token in enumerate(tokens)}

def build_position_dictionary(src_lines, tgt_lines):
    counts = defaultdict(Counter)
    for src_line, tgt_line in zip(src_lines, tgt_lines):
        src_tokens = src_line.split()
        tgt_tokens = tgt_line.split()
        if not src_tokens or not tgt_tokens:
            continue
        for src_index, src_token in enumerate(src_tokens):
            tgt_index = min(round(src_index * (len(tgt_tokens) - 1) / max(1, len(src_tokens) - 1)), len(tgt_tokens) - 1)
            counts[src_token][tgt_tokens[tgt_index]] += 1
    return {src_token: tgt_counts.most_common(1)[0][0] for src_token, tgt_counts in counts.items()}

def translate_bpe(line, dictionary):
    return ' '.join(dictionary.get(token, '<unk>') for token in line.split())

def decode_bpe_lines(lines, model_path):
    if spm is None or not Path(model_path).exists():
        return lines
    processor = spm.SentencePieceProcessor(model_file=str(model_path))
    return [processor.decode(line.split()) for line in lines]

In [3]:
train_src = read_lines(CONFIG['train_src'])
train_tgt = read_lines(CONFIG['train_tgt'])
test_src = read_lines(CONFIG['test_src'])[:CONFIG['eval_limit']]
test_tgt = read_lines(CONFIG['test_tgt'])[:CONFIG['eval_limit']]

dictionary = build_position_dictionary(train_src, train_tgt)
hyp_bpe = [translate_bpe(line, dictionary) for line in test_src]

src_text = decode_bpe_lines(test_src, CONFIG['src_spm'])
ref_text = decode_bpe_lines(test_tgt, CONFIG['tgt_spm'])
hyp_text = decode_bpe_lines(hyp_bpe, CONFIG['tgt_spm'])

bleu = sacrebleu.corpus_bleu(hyp_text, [ref_text])
bleu

BLEU = 4.69 39.8/8.5/2.2/0.6 (BP = 1.000 ratio = 1.073 hyp_len = 19524 ref_len = 18194)

In [4]:
baseline_results = pd.DataFrame([{
    'model': 'dictionary_position_baseline',
    'split': 'test',
    'examples': len(hyp_text),
    'bleu': bleu.score,
    'bleu_summary': str(bleu),
}])
baseline_results.to_csv(OUTPUT_TABLES / 'baseline_results.csv', index=False)
baseline_results

,model,split,examples,bleu,bleu_summary
0,dictionary_position_baseline,test,1000,4.688637,BLEU = 4.69 39.8/8.5/2.2/0.6 (BP = 1.000 ratio...


In [5]:
baseline_samples = pd.DataFrame({
    'arabic_input': src_text[:25],
    'reference_english': ref_text[:25],
    'baseline_output': hyp_text[:25],
})
baseline_samples.to_csv(OUTPUT_TRANSLATIONS / 'baseline_samples.csv', index=False)
baseline_samples

,arabic_input,reference_english,baseline_output
0,قبل عدة سنوات، هنا في تيد، قدّم بيتر سكيلمان م...,"several years ago here at ted, peter skillman ...","before many years, here in ted,,, peter, form,..."
1,والفكرة غاية في البساطة. فريق مكوّن من اربعة ي...,and the idea's pretty simple: teams of four ha...,"and the very in simplicity. a, a of four we to..."
2,يجب ان تكون المارش مالو علي القمة.,the marshmallow has to be on top.,"we to be the the, money, on the."
3,ورغماً عن انها تبدو بسيطة للغاية، الا انها صعب...,"and, though it seems really simple, it's actua...","and the, about it look simple., the it difficu..."
4,لذا فقد فكرت بان هذه فكرة مثيرة، وقمت بتضمينها...,"and so, i thought this was an interesting idea...","so we i that this idea interesting, and to,. i..."
5,وقد كان نجاحاً باهراً.,and it was a huge success.,"and was success, the,."
6,ومنذ ذلك الحين، قمت بعقد حوالي 70 ورشة عمل للت...,"and since then, i've conducted about 70 design...","and that,, i,, about 70 and,, to design throug..."
7,اذاً، في العادة، يبدا معظم الناس بتوجيه انفسهم...,"so, normally, most people begin by orienting t...","so, in,, it most people to add themselves this..."
8,يتحدثون عنها، ويتعرفون علي كيف سيكون شكلها، وي...,"they talk about it, they figure out what it's ...","people about, and know on how it shape it, and..."
9,ثم يقضون بعض الوقت في التخطيط، والتنظيم. انهم ...,"then they spend some time planning, organizing...","and it a some time in the planning, and, organ..."
